# 20 CNN LSTM

Uses past obstacle-clearance strips plus ego/goal sequence features. All outputs are saved to disk for restart-safe experiments.


In [1]:
import gc
import json
import sys
from pathlib import Path

import torch

gc.collect()
try:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception:
    pass

PROJECT_ROOT = Path.home() / "Documents/Thesis"
PIPELINE_ROOT = PROJECT_ROOT / "model_training"
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from training.notebook_workflow import (
    CNNGNNLSTMTrajectoryPredictor,
    CNNGNNLSTMTransformerTrajectoryPredictor,
    CNNGNNTransformerTrajectoryPredictor,
    CNNLSTMTrajectoryPredictor,
    ScanGoalTrajectoryDataset,
    ScanGraphTrajectoryDataset,
    collate_scan,
    collate_scan_graph,
    describe_dataloaders,
    describe_model,
    describe_shared_artifacts,
    device_from_flag,
    evaluate_trajectory_model,
    finish_runtime_report,
    load_or_build_shared_artifacts,
    make_dataloaders,
    prepare_result_dirs,
    run_constant_velocity_baseline,
    save_final_trajectory_evaluation,
    start_runtime_report,
    set_seed,
    timestamp_tag,
    train_trajectory_model,
)


In [2]:
SEED = 42
PAST_LEN = 10
FUTURE_LEN = 5
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
MAX_SAMPLES = None
USE_CPU = False

BATCH_SIZE = 64
EPOCHS = 30
EARLY_STOPPING_PATIENCE = 5
LR = 1e-3
WEIGHT_DECAY = 1e-4

GOAL_DIM = 13
NODE_DIM = 12
EDGE_DIM = 8
HIDDEN_DIM = 128
GRAPH_HIDDEN = 96
DROPOUT = 0.10
MSG_PASSES = 2
TRANSFORMER_HEADS = 4
TRANSFORMER_LAYERS = 2
TRANSFORMER_FF = 256

device = device_from_flag(USE_CPU)
print("Device:", device)


Device: cuda


In [3]:
shared_runtime = start_runtime_report(
    stage_name="shared_data_split",
    output_dir=PIPELINE_ROOT / "results",
    context={"past_len": PAST_LEN, "future_len": FUTURE_LEN, "seed": SEED},
)
set_seed(SEED)
streams, sample_table, split_info, sample_table_path, split_path = load_or_build_shared_artifacts(
    past_len=PAST_LEN,
    future_len=FUTURE_LEN,
    seed=SEED,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
)
print("Sample table:", sample_table_path)
print("Split path:", split_path)
print("Split strategy:", split_info["split_strategy"])
print("Episode count:", split_info["episode_count"])
print("Train / Val / Test samples:", len(split_info["train_indices"]), len(split_info["val_indices"]), len(split_info["test_indices"]))
print("Train episodes:", split_info["train_episode_ids"])
print("Val episodes:", split_info["val_episode_ids"])
print("Test episodes:", split_info["test_episode_ids"])
shared_runtime_summary = finish_runtime_report(
    shared_runtime,
    extra=describe_shared_artifacts(
        streams=streams,
        sample_table=sample_table,
        split_info=split_info,
        sample_table_path=sample_table_path,
        split_path=split_path,
    ),
)
print(json.dumps(shared_runtime_summary, indent=2))


[runtime] shared_data_split start: 2026-05-23T11:23:04 summary=/home/basudeo/Documents/Thesis/model_training/results/runtime/20260523_112304_shared_data_split_runtime_summary.json
Sample table: /home/basudeo/Documents/Thesis/model_training/results/train_ready/sample_table_seed42_past10_future5.json
Split path: /home/basudeo/Documents/Thesis/model_training/results/splits/trajectory_split_seed42_past10_future5.json
Split strategy: episode
Episode count: 8
Train / Val / Test samples: 36405 6082 9219
Train episodes: ['run_model_20260522_203433', 'run_model_20260522_204215', 'run_model_20260523_013946', 'run_model_20260523_015017', 'run_model_20260522_202537']
Val episodes: ['run_model_20260523_013116']
Test episodes: ['run_model_20260522_201207', 'run_model_20260522_201854']
[runtime] shared_data_split done: start=2026-05-23T11:23:04 end=2026-05-23T11:23:08 elapsed_s=4.428 peak_cpu=11.70% peak_rss_mb=1061.44 peak_gpu_alloc_mb=0.00
{
  "stage_name": "shared_data_split",
  "timestamp": "2026

In [4]:
MODEL_SLUG = "cnn_lstm"
TIMESTAMP = timestamp_tag()
result_dir, weight_dir, plot_dir = prepare_result_dirs(MODEL_SLUG)

dataset = ScanGoalTrajectoryDataset(streams, sample_table, PAST_LEN)
train_loader, val_loader, test_loader = make_dataloaders(
    dataset,
    split_info,
    batch_size=BATCH_SIZE,
    collate_fn=collate_scan,
    max_samples=MAX_SAMPLES,
)
loader_stats = describe_dataloaders(train_loader=train_loader, val_loader=val_loader, test_loader=test_loader)

model = CNNLSTMTrajectoryPredictor(
    goal_dim=GOAL_DIM,
    hidden_dim=HIDDEN_DIM,
    cnn_hidden=HIDDEN_DIM,
    future_len=FUTURE_LEN,
    dropout=DROPOUT,
).to(device)
model_stats = describe_model(model)
notebook_runtime = start_runtime_report(
    stage_name=f"{MODEL_SLUG}_notebook_run",
    output_dir=result_dir,
    context={
        "model_slug": MODEL_SLUG,
        "timestamp": TIMESTAMP,
        "past_len": PAST_LEN,
        "future_len": FUTURE_LEN,
        **loader_stats,
        **model_stats,
    },
)

train_out = train_trajectory_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    model_slug=MODEL_SLUG,
    timestamp=TIMESTAMP,
    split_path=split_path,
    sample_table_path=sample_table_path,
    result_dir=result_dir,
    weight_dir=weight_dir,
    plot_dir=plot_dir,
    epochs=EPOCHS,
    patience=EARLY_STOPPING_PATIENCE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    extra_manifest={"family": "scan_goal_cnn_lstm", "future_len": FUTURE_LEN},
    runtime_output_dir=result_dir,
    runtime_context={
        **loader_stats,
        **model_stats,
        **describe_shared_artifacts(
            streams=streams,
            sample_table=sample_table,
            split_info=split_info,
            sample_table_path=sample_table_path,
            split_path=split_path,
        ),
    },
)
test_eval = evaluate_trajectory_model(model, test_loader, device)
notebook_runtime_summary = finish_runtime_report(
    notebook_runtime,
    extra={
        **loader_stats,
        **model_stats,
        "test_sample_count": int(len(test_loader.dataset)),
    },
)
final_metrics = save_final_trajectory_evaluation(
    model_slug=MODEL_SLUG,
    timestamp=TIMESTAMP,
    train_out=train_out,
    test_eval=test_eval,
    split_path=split_path,
    sample_table_path=sample_table_path,
    result_dir=result_dir,
    plot_dir=plot_dir,
    notebook_runtime=notebook_runtime_summary,
)
print(json.dumps(final_metrics, indent=2))


[runtime] cnn_lstm_notebook_run start: 2026-05-23T11:23:08 summary=/home/basudeo/Documents/Thesis/model_training/results/cnn_lstm/runtime/20260523_112308_cnn_lstm_notebook_run_runtime_summary.json
[runtime] cnn_lstm_training start: 2026-05-23T11:23:09 summary=/home/basudeo/Documents/Thesis/model_training/results/cnn_lstm/runtime/20260523_112309_cnn_lstm_training_runtime_summary.json
[cnn_lstm] start: epochs=30 patience=5 train_batches=569 val_batches=96
[cnn_lstm] epoch 01/30 train_loss=0.000623 val_loss=0.000049 val_ADE=0.010000 val_FDE=0.008771 val_RMSE=0.013960 best_ADE=0.010000 status=improved
[cnn_lstm] epoch 02/30 train_loss=0.000526 val_loss=0.000056 val_ADE=0.010267 val_FDE=0.009402 val_RMSE=0.014961 best_ADE=0.010000 status=no_improve(1/5)
[cnn_lstm] epoch 03/30 train_loss=0.000519 val_loss=0.000047 val_ADE=0.009217 val_FDE=0.007654 val_RMSE=0.013651 best_ADE=0.009217 status=improved
[cnn_lstm] epoch 04/30 train_loss=0.000517 val_loss=0.000046 val_ADE=0.008924 val_FDE=0.006136